# Missing Actors in `the_numbers_with_credibility.csv`

This section checks how many movies in `the_numbers_with_credibility.csv` are missing the `actors` list, and outlines options to fill those gaps.[file:98]


In [5]:
import pandas as pd
import os

# BASE_PATH = "data/archive"
NUMBERS_ENRICHED_FILE = os.path.join("the_numbers_with_credibility.csv")

numbers_enriched = pd.read_csv(NUMBERS_ENRICHED_FILE)
numbers_enriched.head()


,rank,release_date,title,budget,domestic_gross,worldwide_gross,title_clean,release_year,movie_id,cast_mean_credibility,cast_max_credibility,cast_actors_with_score,director_mean_credibility,director_max_credibility,directors_with_score,actors
0,1,"Dec 16, 2015",Star Wars Ep. VII: The Force Awakens,533200000,936662225,2056046835,star wars ep. vii: the force awakens,2015.0,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN
1,2,"Apr 23, 2019",Avengers: Endgame,400000000,858373000,2717503922,avengers: endgame,2019.0,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN
2,3,"Dec 9, 2022",Avatar: The Way of Water,400000000,688809501,2322902023,avatar: the way of water,2022.0,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN
3,4,"Dec 17, 2025",Avatar: Fire and Ash,400000000,250295254,859795254,avatar: fire and ash,2025.0,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN
4,5,"May 17, 2025",Mission: Impossible—The Final Reckoning,400000000,197413515,591353074,mission: impossible—the final reckoning,2025.0,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN


## 1. Count movies with missing `actors`

We treat `actors` as missing if the value is either:

- `NaN`, or
- an empty string `""`.

We compute both the **count** and the **share** of such movies.


In [6]:
total_movies = len(numbers_enriched)

is_missing_actors = numbers_enriched["actors"].isna() | (numbers_enriched["actors"].astype(str).str.strip() == "")

missing_actors_count = is_missing_actors.sum()
missing_actors_share = missing_actors_count / total_movies

print("Total movies:", total_movies)
print("Movies with missing actors:", missing_actors_count)
print("Share missing (fraction):", round(missing_actors_share, 3))


Total movies: 6753
Movies with missing actors: 2596
Share missing (fraction): 0.384


## 2. Inspect movies with missing `actors`

To understand possible matching problems (e.g., title/year mismatches), we look at a sample of movies where the `actors` column is missing.


In [7]:
missing_rows = numbers_enriched[is_missing_actors].copy()

missing_rows[["rank", "title", "release_date"]].head(20)


,rank,title,release_date
0,1,Star Wars Ep. VII: The Force Awakens,"Dec 16, 2015"
1,2,Avengers: Endgame,"Apr 23, 2019"
2,3,Avatar: The Way of Water,"Dec 9, 2022"
3,4,Avatar: Fire and Ash,"Dec 17, 2025"
4,5,Mission: Impossible—The Final Reckoning,"May 17, 2025"
7,8,Fast X,"May 17, 2023"
8,9,Solo: A Star Wars Story,"May 23, 2018"
9,10,Avengers: Infinity War,"Apr 25, 2018"
10,11,Pirates of the Caribbean: At World’s End,"May 24, 2007"
12,13,Indiana Jones and the Dial of Destiny,"Jun 28, 2023"


## 3. Options to fill missing `actors`

For movies where `actors` is missing, there are several ways to improve coverage:

1. **TMDb API – `/movie/{movie_id}/credits`**  
   - Many rows already have TMDb `movie_id` (from the join with `movies_metadata.csv`).[web:87]  
   - For those, call the TMDb API endpoint  
     `https://api.themoviedb.org/3/movie/{movie_id}/credits`[web:91]  
     to retrieve a fresh `cast` list and rebuild the `actors` string.

2. **IMDb datasets via ID mapping**  
   - Use `links.csv` from The Movies Dataset to map TMDb IDs to IMDb IDs.[web:87]  
   - Join `title.principals.tsv` + `name.basics.tsv` in the IMDb bulk data to reconstruct cast lists for those IMDb titles.

3. **Wikidata / Wikipedia**  
   - For any remaining unmatched titles, query Wikidata (SPARQL) or scrape individual Wikipedia film pages by title + year to obtain cast lists.

In practice, **re‑matching titles/years and querying the TMDb credits API** usually resolves most missing `actors` entries without heavy scraping.[web:91][web:87]
